In [1]:
# Required Python packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
import copy
from numpy.linalg import multi_dot
from scipy.stats import norm
from scipy.stats import bernoulli
from scipy.optimize import fmin_slsqp as min

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
from statsmodels.api import OLS

from sklearn.linear_model import LassoCV
from yellowbrick.datasets import load_concrete
from yellowbrick.regressor import AlphaSelection

from scipy.interpolate import CubicSpline

import matplotlib.pyplot as plt
import math
import pickle

#Import Date opertative user defined functions
from ipynb.fs.full.user_defined_vik_functions import get_all_monthly_option_expiries, \
                                                     find_last_thurs_date_of_month, \
                                                     prev_workday_if_holiday, find_wkly_expries,\
                                                     date_of_prev_thurs

# Import dataframe naming functions 
from ipynb.fs.full.user_defined_vik_functions import get_mthly_df_name_from_expiry

#Import data loading functions
from ipynb.fs.full.user_defined_vik_functions import load_all_mthly_data

#import traded options parameters and info
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_strikes, generate_mthly_strikes

#Import pricing functions
from ipynb.fs.full.user_defined_vik_functions import BSM_call_vec_with_div, BSM_put_vec_with_div

#Import implied volatility processing functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_iv, gen_interpolated_iv

#Import Risk-free interest rate generation functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_ir

#Import Stochatic process simulation functions
from ipynb.fs.full.user_defined_vik_functions import gbm1W_simulation, generate_covariance_from_correlation

# Import risk free interest rate function
from ipynb.fs.full.user_defined_vik_functions import get_risk_free_rate_from_exact_date


In [11]:
# Stock Index of Interest
stock_ident = "BANKNIFTY"
# stock_ident = "NIFTY"

#Static hedging performed at different moneyness regions 
#i.e. moneyness is used to select the option with nearest moneyness match
# ATM - At the Money, ITM - In the money, OTM - Out of the Money
prod_moneyness = "ATM"

#Product type to hedge: either "CE" or "PE"
prod_type = "CE"
# prod_type = "PE"

# The scope of this code is to hedge one option 
# and scope will be extended to a portfolio
no_of_assets = 1
cor_mat = [[1]]

#Path to refer data
source_path = "/home/jupyter-partha/Vikranth - Chapter 2/"
input_sub_path = "Input Data/mkt_data_covid_region/"
output_sub_path = "Output Data/"
input_data_path = source_path + input_sub_path
output_data_path = source_path + output_sub_path


# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
periods_of_interest = {2019: [8, 9, 10, 11, 12], 2020: [1, 2, 3, 4, 5, 6, 7]}

#List of holidays
holidays_list = [date(2019, 3, 4), date(2019, 3, 21),\
                 date(2019, 4, 17), date(2019, 4, 19), date(2019, 4, 29),\
                 date(2019, 5, 1),\
                 date(2019, 6, 5),\
                 date(2019, 8, 12), date(2019, 8, 15),\
                 date(2019, 9, 2), date(2019, 9, 10), \
                 date(2019, 10, 2), date(2019, 10, 8), date(2019, 10, 21), date(2019, 10, 28), \
                 date(2019, 11, 12), \
                 date(2019, 12, 25), \
                 date(2020, 2, 21), \
                 date(2020, 3, 10), \
                 date(2020, 4, 2), date(2020, 4, 6), date(2020, 4, 10), date(2020,4, 14), \
                 date(2020, 5, 1), date(2020, 5, 25), \
                 date(2020, 10, 2), date(2020, 11, 16), date(2020, 11, 30), date(2020, 12, 25)]

#Simulation Parameters
no_of_paths = 5000

#Number of options
no_opt=1


In [12]:
#########################
#User-defined functions #
#########################

def cubic_func(x, a0, a1, a2, a3):
    y = a0 + a1*x + a2*(np.square(x)) + a3 * np.power(x, 3)
    return y

def generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, \
                                       dict_wkly_expiries_each_month, \
                                       dict_ce_wkly_strikes, dict_pe_wkly_strikes, \
                                       dict_wkly_spots, \
                                       prod_type, prod_moneyness, output_path, \
                                       option_type = "CE", no_of_assets=1, cor_mat=[[1]],\
                                       batch_size_divisor=10, no_of_epochs = 1000, stock_ident="BANKNIFTY"):
    ce_pfl_weights = []
    pe_pfl_weights = [] 
    cash_weights = []
    cubic_polyn_volSurf = pd.read_csv(output_path + "A3_" + stock_ident + "_cubicPolyn_vol_surface_params.csv")
    for each_month in mthly_expiries_list:
        strike = dict_mthly_strikes[each_month.strftime("%d-%b-%Y")]
        weekly_pfl_weights = []
        ce_dict = {}
        pe_dict = {}
        cash_dict = {}
        week_list = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")]
        no_of_weeks = len(week_list)
        
        for week in range(0, no_of_weeks):
            s_t = dict_wkly_spots[each_month.strftime("%d-%b-%Y")][week] 
            K = strike          
            r = np.array([dict_wkly_ir[each_month.strftime("%d-%b-%Y")][week]])
            
            if (week == no_of_weeks-1):
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = each_month
            else:
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week + 1]


            df_polyn_date = cubic_polyn_volSurf[cubic_polyn_volSurf['Date'] == sim_start_date.strftime("%d-%b-%Y")]

            # We define Moneyness as (S/K) in vol surface for both calls and puts
            option_moneyness = float(s_t/float(K))
    
            t_list = np.sort(np.unique(np.array(df_polyn_date['T'])))
            tenor_list = [str(int(round(t* 365))) + "D" for t in t_list]
            print(each_month, " Week: ", week)
            print(tenor_list)
            
            if (len(tenor_list) > 2):
                print("There are more than 2 tenors in the vol surface - Please check!")
            
            df_polyn_coeff = df_polyn_date[df_polyn_date['Tenor'] == tenor_list[0]]
            polyn_coeff_wk = df_polyn_coeff[["a0", "a1", "a2", "a3"]].to_numpy().reshape(-1)

            x_min, x_max = df_polyn_coeff[["x_min", "x_max"]].to_numpy().reshape(-1)

            if (option_moneyness <= x_min):
                vol_list = [cubic_func(x_min, *polyn_coeff_wk)]
            elif (option_moneyness >= x_max):
                vol_list = [cubic_func(x_max, *polyn_coeff_wk)]
            else:
                vol_list = [cubic_func(option_moneyness, *polyn_coeff_wk)]

            sim_stock_mat = gbm1W_simulation(no_of_paths, sim_start_date, sim_end_date, \
                                             no_of_assets, s_t, r, vol_list, cor_mat)
            
            call_strikes = np.sort(np.array(dict_ce_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).reshape(1,-1)
            put_strikes = np.sort(np.array(dict_pe_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).reshape(1,-1)
            no_of_calls = call_strikes.size
            no_of_puts = put_strikes.size
            no_ind_vars = no_of_calls + no_of_puts
                        
            stock_vec = sim_stock_mat[:, 1]
            mny_list = stock_vec / K
            
            r_f = get_risk_free_rate_from_exact_date(sim_start_date)
            dt = float((each_month - sim_end_date).days) / 365
                    
            # Linear interpolation in varaince space across tenors
            if (week != no_of_weeks-1):
                
                # Time variables
                t1 = t_list[0]
                t2 = t_list[1] 
                t12 = dt
                
                # Vol at larger tenor: corresponding to maturity of option to be hedged
                
                df_polyn_coeff = df_polyn_date[df_polyn_date['Tenor'] == tenor_list[1]]
                polyn_coeff_mth = df_polyn_coeff[["a0", "a1", "a2", "a3"]].to_numpy().reshape(-1)
            
                x_min, x_max = df_polyn_coeff[["x_min", "x_max"]].to_numpy().reshape(-1)
                
                vol_t2 = cubic_func(mny_list, *polyn_coeff_mth)
                vol_t2[mny_list <= x_min] = cubic_func(x_min, *polyn_coeff_mth)
                vol_t2[mny_list >= x_max] = cubic_func(x_max, *polyn_coeff_mth)
                #Variance corresponding to tenor t2
                var_t2 = np.square(vol_t2)
                
                #Vol at weekly tenor: corresponding to maturity of options used to hedge
                df_polyn_coeff = df_polyn_date[df_polyn_date['Tenor'] == tenor_list[0]]
                polyn_coeff_wk = df_polyn_coeff[["a0", "a1", "a2", "a3"]].to_numpy().reshape(-1)
                x_min, x_max = df_polyn_coeff[["x_min", "x_max"]].to_numpy().reshape(-1)
                
                vol_t1 = cubic_func(mny_list, *polyn_coeff_wk)
                vol_t1[mny_list <= x_min] = cubic_func(x_min, *polyn_coeff_wk)
                vol_t1[mny_list >= x_max] = cubic_func(x_max, *polyn_coeff_wk)
                #Variance corresponding to tenor t1
                var_t1 = np.square(vol_t1)
                
                #Forward Vol in variance space
                var_t12 = (var_t2 * t2 - var_t1 * t1) / t12
                vol_arr = np.sqrt(var_t12)
            else:
                vol_arr = np.array([0])                 
          
            opt_strike = strike

            if (option_type == "CE" and week != no_of_weeks-1):
                option_value = BSM_call_vec_with_div(stock_vec, opt_strike, r, r_f, vol_arr, dt)
            elif (option_type == "PE" and week != no_of_weeks-1):
                option_value = BSM_put_vec_with_div(stock_vec, opt_strike, r, r_f, vol_arr, dt) 
            elif (option_type == "CE" and week == no_of_weeks-1):
                option_value = np.maximum(stock_vec - opt_strike, 0) 
            else:
                option_value = np.maximum(opt_strike - stock_vec, 0)
    
            option_value = np.asarray(option_value)
            option_value = option_value.reshape(-1, 1) 
            
            y = option_value
            stock_vec = stock_vec.reshape(-1,1)
            x = np.concatenate((np.maximum(stock_vec-call_strikes,0), np.maximum(put_strikes-stock_vec,0)), axis=1) 
            
            
#             print("SKLearn Linear regression - OLS")
#             lin_model = LinearRegression().fit(x, y)
#             pfl_weights = lin_model.coef_
#             pfl_weights = pfl_weights.reshape(-1)
#             cash_weight = lin_model.intercept_
#             print("R-Square: ", lin_model.score(x,y))
#             print("Coefficients:")
#             print(pfl_weights)
#             print(cash_weight)           
            
#             print("SKLearn Linear regression - Ridge")
#             lin_model_ridge = Ridge(alpha=1.0).fit(x, y)
#             pfl_weights = lin_model_ridge.coef_
#             pfl_weights = pfl_weights.reshape(-1)
#             cash_weight = lin_model_ridge.intercept_
#             print("R-Square: ", lin_model_ridge.score(x, y))
#             print("Coefficients:")
#             print(pfl_weights)
#             print(cash_weight)
            
            
            print("SKLearn Linear regression - Lasso")
            lin_model_lasso = Lasso(alpha=1.0).fit(x, y)
            pfl_weights = lin_model_lasso.coef_
            pfl_weights = pfl_weights.reshape(-1)
            cash_weight = lin_model_lasso.intercept_
            print("R-Square: ", lin_model_lasso.score(x, y))
            print("Coefficients:")
            print(pfl_weights)
            print(cash_weight)
            
#          
#             #Find Optimal Alpha
#             alphas = np.array([-0.5, 0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0])
#             model = LassoCV(alphas=alphas)
#             visualizer = AlphaSelection(model)
#             visualizer.fit(x, y)
#             visualizer.show()
            
#             print("Stats Model _ OLS")
#             x = x
#             x = sm.add_constant(x)
#             lin_model2 = OLS(y,x).fit()
#             print(lin_model2.summary())
# #             print(lin_model2.params)
# #           pfl_weights = np.array(lin_model2.params).reshape(-1)
            
            
            ######################################################################################
            ### Portfolio weights of shorter term options from neural network of European options ###
            ######################################################################################
            
            # Call Pfl weights
            call_strikes = call_strikes.reshape(-1) 
            ce_dates_list = np.array([sim_start_date.strftime("%d-%b-%Y") for i in range(0, no_of_calls)]).reshape(-1)
            ce_wt_list = pfl_weights[0:no_of_calls]
            ce_dict = {"Date": ce_dates_list, "Strike": call_strikes, "Pfl_weights": ce_wt_list}
            df_ce = pd.DataFrame(ce_dict)
            ce_pfl_weights.append(df_ce)
    
            
            #Put Pfl Weights
            put_strikes = put_strikes.reshape(-1) 
            pe_dates_list = np.array([sim_start_date.strftime("%d-%b-%Y") for i in range(0, no_of_puts)]).reshape(-1)
            pe_wt_list = pfl_weights[no_of_calls:no_ind_vars]
            pe_dict = {"Date": pe_dates_list, "Strike":put_strikes, "Pfl_weights": pe_wt_list}
            df_pe = pd.DataFrame(pe_dict)
            pe_pfl_weights.append(df_pe)
            
            # Cash component
            cash_date = [sim_start_date.strftime("%d-%b-%Y")]
            cash_dict = {"Date":cash_date, "Pfl_weights":cash_weight} 
            df_cash = pd.DataFrame(cash_dict)
            cash_weights.append(df_cash)
            
            
        df_ce_pfl_weights = pd.concat(ce_pfl_weights, axis=0)
        df_pe_pfl_weights = pd.concat(pe_pfl_weights, axis=0)
        df_cash_pfl_weights = pd.concat(cash_weights, axis=0)

        df_ce_pfl_weights.to_csv(output_path  + "B14_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_CE_Pfl_weights.csv", index = False)
        df_pe_pfl_weights.to_csv(output_path + "B14_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_PE_Pfl_weights.csv", index = False)
        df_cash_pfl_weights.to_csv(output_path  + "B14_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_Cash_Pfl_weights.csv", index = False)
    
    return df_ce_pfl_weights, df_pe_pfl_weights, df_cash_pfl_weights

def output_strikes_hedged(prod_type, prod_moneyness, dict_mthly_strikes, stock_ident, output_path):
    list_of_dates = []
    list_of_prod_type = []
    list_of_strikes = []
    for each_month in dict_mthly_strikes.keys():
        list_of_dates.append(each_month)
        list_of_strikes.append(dict_mthly_strikes[each_month])
        list_of_prod_type.append(prod_type)
    dict_hedged_prod = {"Date":list_of_dates, "prod_type":list_of_prod_type, "Strike":list_of_strikes}
    df_hedged_prod = pd.DataFrame(dict_hedged_prod) 
    df_hedged_prod.to_csv(output_path + "B14_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_strikes_hedged.csv", index = False)
    return(df_hedged_prod)


In [13]:

#Find the monthly strikes of option from mkt data to find the option to be hedged
#Every month, an option is hedged
mthly_expiries_list = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
dict_wkly_expiries_each_month = find_wkly_expries(mthly_expiries_list, holidays_list)

# #Load all monthly mkt data
mthly_mkt_data = load_all_mthly_data(mthly_expiries_list, input_data_path, holidays_list, prod_type_lists=["FUT", "CE", "PE"], stock_ident=stock_ident)

# #Load weekly strikes of short term options used for hedging monthly options 
dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots = generate_weekly_strikes(dict_wkly_expiries_each_month, input_data_path, stock_ident = "BANKNIFTY")     

#Load monthly strikes of interest as of required moneyness on the begining of month
dict_mthly_strikes = generate_mthly_strikes(mthly_mkt_data, prod_moneyness, prod_type, holidays_list, stock_ident=stock_ident)

#Load weekly risk free interest rates from futures
#This function to be changed based on updates on risk-free-interest - now kept it as cosntant
dict_wkly_ir = generate_weekly_ir(dict_wkly_expiries_each_month, mthly_mkt_data, stock_ident)

# # #Load Static portfolio weights built by neural network
ce_pfl_wts, pe_pfl_wts, cash_pfl_wts = generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, dict_wkly_expiries_each_month, \
                                                        dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots, \
                                                        prod_type, prod_moneyness, output_data_path, \
                                                        option_type = prod_type, no_of_assets=no_of_assets, \
                                                        cor_mat=cor_mat, stock_ident=stock_ident) 

# # # The strike correspnding to the option that is hedged
df_hedged_prod = output_strikes_hedged(prod_type, prod_moneyness, dict_mthly_strikes, stock_ident, output_data_path)



2019-08-29  Week:  0
['7D', '35D']
SKLearn Linear regression - Lasso
R-Square:  0.9992327992987804
Coefficients:
[-0.06566047  0.00343632  0.00335589  0.00716797  0.00773464  0.00695879
  0.00597732  0.00509774  0.0043295   0.00366818  0.003106    0.00269438
  0.00235022  0.00188223  0.00082002  0.00508095  0.23545632  0.08562688
  0.          0.01103535  0.02852616  0.02685786  0.01252108  0.00135796
  0.01382681  0.04687145  0.05070694  0.00684409  0.          0.
  0.        ]
[49.23939609]
2019-08-29  Week:  1
['7D', '28D']


/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 6700.943478444882, tolerance: 4336.717516784021
  positive)
/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 127434.45273168045, tolerance: 21418.31724036575
  positive)


SKLearn Linear regression - Lasso
R-Square:  0.9982671929371468
Coefficients:
[-0.17346858  0.02667809  0.0097208   0.01268896  0.01445576  0.01362811
  0.01240659  0.01116889  0.00992525  0.00840333  0.00747759  0.00724516
  0.00686335  0.00443418  0.          0.02075016  0.5809318  -0.31808076
  0.03830355  0.02506831  0.02153799  0.03607759  0.04285335  0.02811376
  0.00481175 -0.00553683  0.03240526  0.09264084  0.09596854  0.01832828
 -0.00076256]
[134.8781517]
2019-08-29  Week:  2
['6D', '21D']
SKLearn Linear regression - Lasso
R-Square:  0.9999831754876657
Coefficients:
[-0.2176561   0.0234549   0.01745278  0.01960067  0.02028059  0.0197524
  0.0180995   0.01690606  0.01475222  0.00875066  0.0039465   0.02831678
  0.01863796  0.10478396  0.12844084  0.          0.0422967   0.03416814
  0.03224007  0.04853749  0.04975204  0.02971603  0.02925398  0.05593179
  0.05486871  0.01424764  0.00522819  0.11697173  0.17545684  0.03818869
  0.0049157   0.00430369  0.00232459]
[156.67272045]

/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 27222.711998876195, tolerance: 9132.817519082917
  positive)
/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 24387.999551585635, tolerance: 10875.532005138086
  positive)


SKLearn Linear regression - Lasso
R-Square:  0.9992738849091389
Coefficients:
[-9.86899353e-02  1.95299007e-02  1.57631931e-02  1.18386258e-02
  8.20940270e-03  6.86574411e-03  5.99058616e-03  5.24150849e-03
  4.32712365e-03  3.67669888e-03  3.00948969e-03  2.67708871e-03
  2.56410167e-03  1.85648365e-03  3.55944893e-05  0.00000000e+00
  5.68451144e-03  3.13603774e-01 -1.09048887e-03  7.56531799e-02
  2.39593948e-02  2.53029746e-02  1.64822520e-02  2.01575263e-02
  2.86095513e-02  2.96177548e-02  1.96047705e-02  5.62229024e-03
  1.25884003e-03  1.85438682e-02  4.71314920e-02  5.37240143e-02
  1.86937484e-02  4.93812638e-03]
[53.7048681]
2019-09-26  Week:  2
['7D', '14D']
SKLearn Linear regression - Lasso
R-Square:  0.9699961401808646
Coefficients:
[-2.59345702e-03 -0.00000000e+00 -0.00000000e+00  0.00000000e+00
  8.18644140e-04  5.54687710e-04  3.67246542e-04  2.51385818e-04
  1.75550805e-04  1.35026541e-04  1.26220603e-04  1.01005990e-04
  0.00000000e+00  0.00000000e+00  3.34735919e-0

...

R-Square:  0.9999737953197267
Coefficients:
[-0.21233482  0.02818386  0.01661641  0.01268203  0.01112235  0.01021053
  0.00969817  0.00905422  0.00834059  0.00760307  0.00711429  0.00682503
  0.00726162  0.00821866  0.01918766  0.02108267  0.00747904  0.01551816
  0.05221085  0.07019076  0.09398136  0.1033986   0.12871436  0.10233353
 -0.01397287 -0.0032317   0.01880549  0.03608856  0.04434006  0.05617776
  0.00484649 -0.00774757 -0.00921251  0.00597944  0.03620265  0.06373041
  0.08171922  0.06967369  0.03174038  0.01007146]
[250.59735191]
2020-06-25  Week:  1
['7D', '21D']
SKLearn Linear regression - Lasso


/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 61429.10071340774, tolerance: 28704.887219591452
  positive)


R-Square:  0.9994155096731496
Coefficients:
[-4.16797895e-02  6.83331276e-03  6.58178872e-03  5.83762959e-03
  5.71233696e-03  5.52862798e-03  4.64764748e-03  2.58039027e-03
  1.34978954e-03  8.36100624e-04  7.33683274e-04  7.48741438e-04
  6.98183788e-04  8.88210480e-04  6.29410345e-04  1.15860501e-04
 -8.51493897e-04 -4.52296628e-03 -5.50852779e-03 -2.26550079e-03
  4.50400988e-03  2.96818665e-03  1.01297756e-03  2.02608220e-03
  3.36568031e-01  4.91333990e-02  1.37237138e-01  9.06946346e-02
  5.47440305e-02  1.73948121e-02  1.19546070e-02  1.04888884e-02
  7.93710346e-03  8.66834991e-03  4.77212658e-03  6.93230661e-03
  9.30897085e-03  2.37615053e-02  1.29394998e-02  1.03571357e-02
  5.85714897e-03]
[32.6251149]
2020-06-25  Week:  2
['7D', '14D']
SKLearn Linear regression - Lasso


/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 40982.22057135622, tolerance: 23251.899514561414
  positive)


R-Square:  0.9993470136344679
Coefficients:
[-1.75236379e-02  2.75124759e-03  3.07893109e-03  2.39846756e-03
  2.02166688e-03  1.80575954e-03  1.53598268e-03  1.09673951e-03
  6.17399704e-04  3.67769718e-04  2.40891904e-04  1.80504892e-04
  1.58936698e-04  1.55552176e-04  1.58395199e-04  1.66120180e-04
  3.42982993e-04  4.07961996e-04  6.61783068e-05  0.00000000e+00
 -2.26047085e-04 -4.94584173e-03  3.64730699e-03  3.56538824e-01
  8.07389023e-02  1.56585539e-01  1.00181031e-01  6.56939281e-02
  2.84481465e-02  5.85259991e-03  8.93426489e-03  3.30449465e-03
  6.23770216e-03  4.70913741e-03  6.16364214e-03  6.93837458e-03
  6.60085564e-03  4.80677162e-03  2.02124468e-03  3.92060954e-04]
[14.7505936]
2020-06-25  Week:  3
['7D']
SKLearn Linear regression - Lasso
R-Square:  0.9999994413201237
Coefficients:
[-1.02266324e-03  1.36542538e-04  3.73394364e-04  2.43468549e-04
  1.45495802e-04  8.59040653e-05  4.16750272e-05  4.10246674e-06
  0.00000000e+00  0.00000000e+00  0.00000000e+00 -0.0000